In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

In [2]:
class FocalLoss(nn.Module):
    """Categorical Focal Cross-Entropy Loss (Nawaz et al. 2023).
    Combines class weights (alpha) with a focusing factor (gamma)
    to down-weight easy majority-class examples and focus training
    on hard minority-class examples (Heartbleed, SQL_Injection…).
    gamma=0 reduces to standard weighted CrossEntropyLoss.
    gamma=2 is the value recommended in the literature.
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha     = alpha      # per-class weight tensor
        self.gamma     = gamma      # focusing parameter
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Per-sample cross-entropy with optional class weights
        ce_loss = F.cross_entropy(
            inputs, targets,
            weight=self.alpha,
            reduction='none'
        )
        # pt = predicted probability of the correct class
        pt = torch.exp(-ce_loss)
        # Focal term: near 0 for easy examples, near 1 for hard examples
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss.sum()

print("FocalLoss defined.")

FocalLoss defined.


In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length=512, dropout=0.5):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() *
            (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k       = d_model // num_heads
        self.num_heads = num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        B, T, D = x.size()
        return x.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        B, _, T, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(B, T, self.num_heads * d_k)

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask, -1e9)
        attn = torch.softmax(scores, dim=-1)
        return self.W_o(self.combine_heads(torch.matmul(attn, V)))


In [5]:
class PointWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.5):
        super().__init__()
        self.fc1     = nn.Linear(d_model, d_ff)
        self.fc2     = nn.Linear(d_ff, d_model)
        self.relu    = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.dropout(self.relu(self.fc1(x))))

In [6]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn       = PointWiseFeedForward(d_model, d_ff, dropout)
        self.norm1     = nn.LayerNorm(d_model)
        self.norm2     = nn.LayerNorm(d_model)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Multi-Head Self-Attention + Add & LayerNorm
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        # FFN + Add & LayerNorm
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x

In [7]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.masked_self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn       = MultiHeadAttention(d_model, num_heads)
        self.ffn              = PointWiseFeedForward(d_model, d_ff, dropout)
        self.norm1            = nn.LayerNorm(d_model)
        self.norm2            = nn.LayerNorm(d_model)
        self.norm3            = nn.LayerNorm(d_model)
        self.dropout          = nn.Dropout(dropout)

    def forward(self, x, enc_output, tgt_mask=None):
        # 1. Masked Self-Attention + Add & LayerNorm
        x = self.norm1(x + self.dropout(
            self.masked_self_attn(x, x, x, tgt_mask)
        ))
        # 2. Cross-Attention (Q=decoder, K=V=encoder) + Add & LayerNorm
        x = self.norm2(x + self.dropout(
            self.cross_attn(x, enc_output, enc_output, None)
        ))
        # 3. FFN + Add & LayerNorm
        x = self.norm3(x + self.dropout(self.ffn(x)))
        return x

In [8]:
class TransformerNIDS(nn.Module):
    def __init__(self, input_dim, num_classes, d_model=32, num_heads=4,
                 num_encoder_layers=6, num_decoder_layers=6,
                 d_ff=1024, dropout=0.5, max_seq_length=512):
        super().__init__()
        self.encoder_embedding = nn.Linear(input_dim, d_model)
        self.decoder_embedding = nn.Linear(input_dim, d_model)
        self.pos_encoding      = PositionalEncoding(d_model, max_seq_length, dropout)
        self.encoder_layers    = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_encoder_layers)
        ])
        self.decoder_layers    = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_decoder_layers)
        ])
        self.fc_out  = nn.Linear(d_model, num_classes)
        self.softmax = nn.Softmax(dim=-1)
        self.dropout = nn.Dropout(dropout)

    def _causal_mask(self, seq_len, device):
        return torch.triu(
            torch.ones(seq_len, seq_len, device=device), diagonal=1
        ).bool()

    def forward(self, x):
        # Encoder
        src = self.pos_encoding(self.encoder_embedding(x))
        enc_output = src
        for layer in self.encoder_layers:
            enc_output = layer(enc_output)

        # Decoder
        tgt = self.pos_encoding(self.decoder_embedding(x))
        tgt_mask = self._causal_mask(x.size(1), x.device)
        dec_output = tgt
        for layer in self.decoder_layers:
            dec_output = layer(dec_output, enc_output, tgt_mask)

        # Classification
        out = dec_output.mean(dim=1)
        return self.softmax(self.fc_out(out))

In [9]:
# ── Load saved CSVs ──────────────────────────────────────────────
train_df = pd.read_parquet("processed/train.parquet")
val_df   = pd.read_parquet("processed/val.parquet")
test_df  = pd.read_parquet("processed/test.parquet")

# ── Encode labels ────────────────────────────────────────────────
# Fit on all splits so val/test labels never cause unseen-label errors
le = LabelEncoder()
le.fit(np.concatenate([
    train_df['Label'].values,
    val_df['Label'].values,
    test_df['Label'].values
]))

X_train = torch.tensor(train_df.drop(columns=['Label']).values, dtype=torch.float32).unsqueeze(1)
y_train = torch.tensor(le.transform(train_df['Label']), dtype=torch.long)

X_val   = torch.tensor(val_df.drop(columns=['Label']).values, dtype=torch.float32).unsqueeze(1)
y_val   = torch.tensor(le.transform(val_df['Label']), dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val),   batch_size=128, shuffle=False)

# ── Device ───────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device      : {device}")

# ── Class weights ────────────────────────────────────────────────
# Algorithm-level imbalance correction: rarer classes get higher weights
y_train_raw = train_df['Label'].values
classes_raw = np.unique(y_train_raw)
raw_weights = compute_class_weight('balanced', classes=classes_raw, y=y_train_raw)
class_weights = torch.tensor(
    [raw_weights[np.where(classes_raw == c)[0][0]] for c in le.classes_],
    dtype=torch.float32
).to(device)

# ── Model setup ──────────────────────────────────────────────────
input_dim   = X_train.shape[-1]
num_classes = len(le.classes_)

model     = TransformerNIDS(input_dim=input_dim, num_classes=num_classes).to(device)
# FocalLoss replaces CrossEntropyLoss — combines SMOTE (data-level)
# with class weights + focal term (algorithm-level)
criterion = FocalLoss(alpha=class_weights, gamma=2.0)
optimizer = torch.optim.SGD(model.parameters(), lr=5e-4, momentum=0.9)

print(f"Input dim   : {input_dim}")
print(f"Num classes : {num_classes}")
print(f"Classes     : {list(le.classes_)}")
print(f"Parameters  : {sum(p.numel() for p in model.parameters()):,}")
print(f"Loss        : FocalLoss(gamma=2.0, weighted=True)")

Device      : cpu
Input dim   : 69
Num classes : 15
Classes     : ['Benign', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SQL_Injection', 'SSH-Patator', 'Web_Brute_Force', 'XSS']
Parameters  : 882,031
Loss        : FocalLoss(gamma=2.0, weighted=True)


In [10]:
EPOCHS = 25
model  = model.to(device)

for epoch in range(EPOCHS):
    # ── Train ────────────────────────────────────
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = model(X_batch)
        loss   = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        train_loss    += loss.item()
        train_correct += (output.argmax(1) == y_batch).sum().item()
        train_total   += len(y_batch)

    # ── Validate ─────────────────────────────────
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            output    = model(X_batch)
            loss      = criterion(output, y_batch)
            val_loss    += loss.item()
            val_correct += (output.argmax(1) == y_batch).sum().item()
            val_total   += len(y_batch)

    print(
        f"Epoch [{epoch+1:2d}/{EPOCHS}] "
        f"Train Loss: {train_loss/len(train_loader):.4f} "
        f"Train Acc: {train_correct/train_total*100:.2f}% | "
        f"Val Loss: {val_loss/len(val_loader):.4f} "
        f"Val Acc: {val_correct/val_total*100:.2f}%"
    )

KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# test_df already loaded above — no need to re-read
X_test      = torch.tensor(test_df.drop(columns=['Label']).values, dtype=torch.float32).unsqueeze(1)
y_test      = torch.tensor(le.transform(test_df['Label']), dtype=torch.long)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=128, shuffle=False)

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        output  = model(X_batch)
        preds   = output.argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(y_batch.numpy())

# ── Classification report ────────────────────────────────────────
print(classification_report(
    all_labels, all_preds,
    target_names=le.classes_,
    zero_division=0
))

# ── Confusion matrix ─────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(14, 11))
sns.heatmap(
    cm,
    annot=True, fmt='d',
    xticklabels=le.classes_,
    yticklabels=le.classes_,
    cmap='Blues'
)
plt.title('Confusion Matrix — RTIDS on CICIDS-2017')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print("Confusion matrix saved to confusion_matrix.png")